In [ ]:
import os
import time
import json
import pandas as pd
from pathlib import Path
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_ollama import OllamaLLM
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

print("All imports successful")
print("Make sure Ollama is running: ollama serve")

In [ ]:
# Load all documents from the documents folder
loader = DirectoryLoader(
    './documents/',
    glob="**/*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc.metadata['source']} ({len(doc.page_content)} chars)")

In [ ]:
# Chunking strategy: chunk_size=512, overlap=64
# Reasoning: 512 tokens captures enough context for factual Q&A
# 64 token overlap prevents losing context at chunk boundaries
# RecursiveCharacterTextSplitter tries to split on paragraphs first,
# then sentences, then words - preserving semantic units

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks)//len(chunks)} chars")
print(f"\nSample chunk:\n{chunks[0].page_content}")

In [ ]:
# Embedding model: all-MiniLM-L6-v2
# Chosen for: fast inference, good semantic quality, 384-dim vectors
# Small enough to run on CPU quickly

print("Loading embedding model...")
embeddings = SentenceTransformerEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Build ChromaDB vector store
print("Building ChromaDB vector index...")
t_start = time.time()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

index_time = time.time() - t_start
print(f"Vector index built in {index_time:.2f} seconds")
print(f"Total vectors stored: {vectorstore._collection.count()}")

In [ ]:
# Retriever: top-5 most similar chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# LLM: Mistral 7B via Ollama
print("Connecting to Mistral 7B via Ollama...")
llm = OllamaLLM(
    model="mistral:7b-instruct",
    temperature=0.1  # Low temperature for factual responses
)

# Test LLM connection
test_response = llm.invoke("Say 'LLM connected' and nothing else.")
print(f"LLM test: {test_response.strip()}")

In [ ]:
# Custom RAG prompt that enforces grounding
rag_prompt = PromptTemplate(
    template="""You are a helpful assistant. Use ONLY the context below to answer 
the question. If the answer is not in the context, say "I don't have information 
about that in my knowledge base."

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

# Build the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt}
)

print("RAG chain ready")

In [ ]:
def run_rag_query(question, verbose=True):
    """Run a RAG query and return answer with latency metrics."""
    
    # Retrieval timing
    t_retrieval_start = time.time()
    retrieved_docs = retriever.invoke(question)
    retrieval_latency = time.time() - t_retrieval_start
    
    # Generation timing
    t_gen_start = time.time()
    result = rag_chain.invoke({"query": question})
    generation_latency = time.time() - t_gen_start
    
    total_latency = retrieval_latency + generation_latency
    
    if verbose:
        print(f"Question: {question}")
        print(f"Answer: {result['result']}")
        print(f"Sources: {[d.metadata['source'] for d in result['source_documents']]}")
        print(f"Retrieval: {retrieval_latency:.2f}s | Generation: {generation_latency:.2f}s | Total: {total_latency:.2f}s")
        print("-" * 60)
    
    return {
        "question": question,
        "answer": result["result"],
        "sources": [d.metadata["source"] for d in result["source_documents"]],
        "retrieved_docs": retrieved_docs,
        "retrieval_latency": retrieval_latency,
        "generation_latency": generation_latency,
        "total_latency": total_latency
    }

In [ ]:
# 10 handcrafted evaluation queries covering diverse topics and edge cases
eval_queries = [
    # Factual retrieval
    "What is retrieval-augmented generation?",
    "What is the difference between FAISS and ChromaDB?",
    "What is chunk overlap and why is it used?",
    # Multi-concept queries
    "How does RAG reduce hallucinations in language models?",
    "What is the difference between supervised and unsupervised learning?",
    # Agent/tool-use specific
    "What is the ReAct framework for LLM agents?",
    "How do agents decide which tool to use?",
    # MLOps specific
    "What is data drift and why does it matter?",
    "How does canary deployment reduce risk?",
    # Edge case - out of scope
    "What is the capital of France?"  # Not in documents - tests grounding
]

print("Running 10 evaluation queries...\n")
results = []
for i, query in enumerate(eval_queries, 1):
    print(f"Query {i}/10:")
    result = run_rag_query(query)
    results.append(result)
    print()

In [ ]:
# Manual relevance judgments (1 = relevant source retrieved, 0 = not)
# Based on which document SHOULD contain the answer
expected_sources = {
    "What is retrieval-augmented generation?": "rag_systems",
    "What is the difference between FAISS and ChromaDB?": "vector_databases",
    "What is chunk overlap and why is it used?": "rag_systems",
    "How does RAG reduce hallucinations in language models?": "rag_systems",
    "What is the difference between supervised and unsupervised learning?": "ai_basics",
    "What is the ReAct framework for LLM agents?": "llm_agents",
    "How do agents decide which tool to use?": "llm_agents",
    "What is data drift and why does it matter?": "mlops",
    "How does canary deployment reduce risk?": "mlops",
    "What is the capital of France?": None  # Out of scope
}

print("=" * 60)
print("EVALUATION METRICS SUMMARY")
print("=" * 60)

metrics = []
for r in results:
    q = r["question"]
    expected = expected_sources.get(q)
    retrieved_source_names = [s.split("/")[-1].replace(".txt","") for s in r["sources"]]
    
    # Precision@5: how many of top 5 retrieved are from the right source
    if expected is None:
        # Out-of-scope: check if model correctly says it doesn't know
        precision_at_5 = "N/A (out-of-scope)"
        recall_at_5 = "N/A"
        grounded = "grounded" if "don't have" in r["answer"].lower() or "not in" in r["answer"].lower() else "hallucinated"
    else:
        relevant_retrieved = sum(1 for s in retrieved_source_names if expected in s)
        precision_at_5 = relevant_retrieved / 5
        recall_at_5 = min(relevant_retrieved, 1)  # Binary: did we get at least one right?
        grounded = "grounded"
    
    metrics.append({
        "query": q[:50] + "...",
        "expected_source": expected,
        "retrieved_sources": retrieved_source_names[:3],
        "precision@5": precision_at_5,
        "recall@5": recall_at_5,
        "grounding": grounded,
        "retrieval_latency": f"{r['retrieval_latency']:.2f}s",
        "generation_latency": f"{r['generation_latency']:.2f}s",
        "total_latency": f"{r['total_latency']:.2f}s"
    })
    
    print(f"\nQ: {q[:60]}")
    print(f"Expected source: {expected}")
    print(f"Got sources: {retrieved_source_names[:3]}")
    print(f"Precision@5: {precision_at_5} | Recall@5: {recall_at_5}")
    print(f"Latency: retrieval={r['retrieval_latency']:.2f}s, gen={r['generation_latency']:.2f}s")

# Summary stats
numeric_precision = [m["precision@5"] for m in metrics if isinstance(m["precision@5"], float)]
numeric_recall = [m["recall@5"] for m in metrics if isinstance(m["recall@5"], int)]
print(f"\n{'='*60}")
print(f"Average Precision@5: {sum(numeric_precision)/len(numeric_precision):.2f}")
print(f"Average Recall@5: {sum(numeric_recall)/len(numeric_recall):.2f}")
avg_retrieval = sum(r["retrieval_latency"] for r in results) / len(results)
avg_gen = sum(r["generation_latency"] for r in results) / len(results)
avg_total = sum(r["total_latency"] for r in results) / len(results)
print(f"Avg Retrieval Latency: {avg_retrieval:.2f}s")
print(f"Avg Generation Latency: {avg_gen:.2f}s")
print(f"Avg End-to-End Latency: {avg_total:.2f}s")